In [1]:
import torch

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 0. Choosing "teacher" model

In [2]:
checkpoint = "bert-base-uncased"
task_name  = "stsb"

# 1. Loading our mrpc part of the GLUE dataset

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", task_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 5749
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1379
    })
})

In [5]:
raw_datasets['train']

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 5749
})

In [6]:
from datasets import DatasetDict

In [7]:
raw_datasets = DatasetDict({
    "train": raw_datasets['train'],
    "validation": raw_datasets['validation'],
    "test": raw_datasets['test']
})

In [8]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 5749
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1379
    })
})

In [9]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'A plane is taking off.',
 'sentence2': 'An air plane is taking off.',
 'label': 5.0,
 'idx': 0}

In [10]:
raw_train_dataset[5]['sentence1']

'Some men are fighting.'

In [11]:
raw_train_dataset[5]['sentence2']

'Two men are fighting.'

In [12]:
raw_train_dataset[5]['idx']

5

In [13]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': Value('float32'),
 'idx': Value('int32')}

# 2. Preprocess

In [14]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

Map: 100%|██████████| 1379/1379 [00:00<00:00, 13261.98 examples/s]


DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5749
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1379
    })
})

# 3. Preparing for Training

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [17]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=32, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=32, collate_fn=data_collator
)

In [18]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([32]),
 'input_ids': torch.Size([32, 66]),
 'token_type_ids': torch.Size([32, 66]),
 'attention_mask': torch.Size([32, 66])}

# 4. Loading model

In [19]:
import sys
sys.path.insert(0, '../')

In [20]:
from Bert_model.modeling_bert import BertForSequenceClassification

In [21]:
num_labels = 1

In [22]:
model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    problem_type="regression",
    # id2label=id2label,
    # label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
model.set_use_module_grafting(False)
model.set_use_scc_status(False)

In [24]:
teacher_model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    problem_type="regression",
    # id2label=id2label,
    # label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [25]:
teacher_model.set_use_module_grafting(False)
teacher_model.set_use_scc_status(False)

### Load Teacher Model

In [26]:
load_path = '../glue_fine_tune/weights/'
best_weight = torch.load(load_path + f'bert-{task_name}.pt', map_location='cpu')
teacher_model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [27]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
teacher_model.to(device)

device

device(type='cuda')

### Take first six layers as student model

In [28]:
model.bert.encoder.layer = model.bert.encoder.layer[:6]

In [29]:
len(model.bert.encoder.layer)

6

In [30]:
model.to(device)

device

device(type='cuda')

# 5. Learning Rate and Leanring Scheduler Setup

In [31]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=3e-5)

In [32]:
from train_eval_func import set_lr_scheduler

num_epochs = 10
num_training_steps = num_epochs * len(train_dataloader)

lr_scheduler = set_lr_scheduler(
    optimizer          = optimizer,
    num_training_steps = num_training_steps
)

print(num_training_steps)

1800


In [33]:
patience=3

In [34]:
student_layer_ids = [1,2,3,4,5,6]

In [35]:
teacher_layer_ids = [1,2,4,6,8,10,12]

In [36]:
import os

In [37]:
save_path = os.path.join(f'./weights/6 layers top/{task_name}.pt')

# 6. Training

In [38]:
from train_eval_func import train

In [39]:
train_stats = train(
    model,
    train_dataloader,
    val_dataloader,
    task_name,
    device,
    num_epochs=num_epochs,
    lr=lr_scheduler,
    patience=patience,
    use_early_stopping=True,
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
    save_path=save_path,
    display_epoch_iter=True,
    teacher_model=teacher_model,
    student_layer_ids=student_layer_ids,
    teacher_layer_ids=student_layer_ids,
    regression=True
)

Training:   0%|          | 0/1800 [00:00<?, ?it/s]

Knowledge distillation used!


/home/abhinavl/work/Layer_Graft/layer-grafting-publication/scratch_train/../train_eval_func.py:240: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  total_loss += float(loss)
Training:  10%|▉         | 179/1800 [00:15<02:03, 13.13it/s]

✓ Saved checkpoint (best loss: 0.5557)

<----------------- Epoch 1 ----------------->
Loss: 0.51, Training Metrics:
  pearson: 0.7479
  spearmanr: 0.7077
Validation Loss: 0.56, Validation Metrics:
  pearson: 0.8749
  spearmanr: 0.8743
Elapsed Time: 16.4594 sec


Training:  20%|█▉        | 359/1800 [00:38<01:55, 12.53it/s]

...


<----------------- Epoch 8 ----------------->
Loss: 0.06, Training Metrics:
  pearson: 0.9781
  spearmanr: 0.9753
Validation Loss: 0.51, Validation Metrics:
  pearson: 0.8826
  spearmanr: 0.8788
Elapsed Time: 17.9281 sec


Training:  90%|████████▉ | 1619/1800 [03:07<00:14, 12.55it/s]


<----------------- Epoch 9 ----------------->
Loss: 0.06, Training Metrics:
  pearson: 0.9796
  spearmanr: 0.9771
Validation Loss: 0.5, Validation Metrics:
  pearson: 0.8838
  spearmanr: 0.8802
Elapsed Time: 16.92 sec


Training: 100%|██████████| 1800/1800 [03:33<00:00,  8.44it/s]

✓ Saved checkpoint (best loss: 0.5052)

<----------------- Epoch 10 ----------------->
Loss: 0.05, Training Metrics:
  pearson: 0.9822
  spearmanr: 0.9801
Validation Loss: 0.51, Validation Metrics:
  pearson: 0.8842
  spearmanr: 0.8805
Elapsed Time: 16.852 sec

Total Training Time: 213.27 sec
Chosen Primary Metric: pearson
Best Metric Run: 0.8842


# 7. Evaluation

In [40]:
best_checkpoint = torch.load(save_path)

In [41]:
model.load_state_dict(best_checkpoint['model_state_dict'])

<All keys matched successfully>

In [42]:
from train_eval_func import eval_loop

In [43]:
eval_loop(model, val_dataloader, task_name, device, regression=True)

({'pearson': 0.8842038602884994, 'spearmanr': 0.880503279459906},
 0.505236158662654)

In [44]:
eval_loop(teacher_model, val_dataloader, task_name, device, regression=True)

({'pearson': 0.900084701022039, 'spearmanr': 0.8962584593578651},
 0.4520208261748578)